In [9]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

In [10]:
# import os
# import numpy as np
# import tensorflow as tf
# import keras
# from keras import layers, models
# from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
# from keras.utils import image_dataset_from_directory

### Config

In [11]:
DATASET_DIR  = "./dataset_split"          
MODEL_DIR    = "./model"
MODEL_PATH   = os.path.join(MODEL_DIR, "pneumonia_densenet.keras")  
IMG_SIZE     = (224, 224)
BATCH_SIZE   = 32
EPOCHS       = 25
LEARNING_RATE = 1e-4

os.makedirs(MODEL_DIR, exist_ok=True)

### Data Generators

In [12]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=10,       # Slight rotation is okay
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=False,   # STRICTLY FALSE for X-Rays
    fill_mode="nearest",
)

val_test_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_gen = train_datagen.flow_from_directory(
    os.path.join(DATASET_DIR, "train"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",   
    shuffle=True,
)

val_gen = val_test_datagen.flow_from_directory(
    os.path.join(DATASET_DIR, "val"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False,
)

test_gen = val_test_datagen.flow_from_directory(
    os.path.join(DATASET_DIR, "test"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False,
)

Found 4099 images belonging to 2 classes.
Found 877 images belonging to 2 classes.
Found 880 images belonging to 2 classes.


### Calculating class weights for imbalance

In [13]:
normal_count = 1341
pneumonia_count = 3875
total_images = normal_count + pneumonia_count

weight_for_0 = (1 / normal_count) * (total_images / 2.0)
weight_for_1 = (1 / pneumonia_count) * (total_images / 2.0)
class_weights = {0: weight_for_0, 1: weight_for_1}

print(f"Applying Class Weights: Normal: {weight_for_0:.2f}, Pneumonia: {weight_for_1:.2f}")

Applying Class Weights: Normal: 1.94, Pneumonia: 0.67


### Model Training using Transfer learning with DenseNet121

In [14]:
def build_model():
    base_model = tf.keras.applications.DenseNet121(
        input_shape=(*IMG_SIZE, 3),
        include_top=False,      
        weights="imagenet",     
    )
    base_model.trainable = False  

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(1, activation="sigmoid"),  
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=[
            "accuracy", 
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.Precision(name="precision"), # Added
            tf.keras.metrics.Recall(name="recall")        # Added
        ],
    )

    return model, base_model

model, base_model = build_model()

### Callbacks

In [15]:
callbacks = [
    ModelCheckpoint(
        filepath=MODEL_PATH,
        monitor="val_auc", # Monitoring AUC is better than accuracy for imbalance
        save_best_only=True,
        mode="max",
        verbose=1,
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
]

### Training with Frozen base

In [16]:
print("\n[PHASE 1] Training classifier head...")
history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    class_weight=class_weights, # Applying the penalty weights here
    callbacks=callbacks,
)


[PHASE 1] Training classifier head...
Epoch 1/25
129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7513 - auc: 0.7962 - loss: 0.5346 - precision: 0.8742 - recall: 0.7774
Epoch 1: val_auc improved from None to 0.97335, saving model to ./model\pneumonia_densenet.keras

Epoch 1: finished saving model to ./model\pneumonia_densenet.keras
129/129 ━━━━━━━━━━━━━━━━━━━━ 371s 3s/step - accuracy: 0.8002 - auc: 0.8915 - loss: 0.4214 - precision: 0.9282 - recall: 0.7870 - val_accuracy: 0.7662 - val_auc: 0.9734 - val_loss: 0.5013 - val_precision: 0.9977 - val_recall: 0.6812 - learning_rate: 1.0000e-04
Epoch 2/25
129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8631 - auc: 0.9535 - loss: 0.2863 - precision: 0.9708 - recall: 0.8401
Epoch 2: val_auc improved from 0.97335 to 0.98755, saving model to ./model\pneumonia_densenet.keras

Epoch 2: finished saving model to ./model\pneumonia_densenet.keras
129/129 ━━━━━━━━━━━━━━━━━━━━ 227s 2s/step - accuracy: 0.8724 - auc: 0.9571 - loss: 0.2746 - prec

### fine tune top layer

In [17]:
print("\n[PHASE 2] Fine-tuning top layers...")
base_model.trainable = True

# Unfreeze only the last block of DenseNet
for layer in base_model.layers[:-40]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"), 
             tf.keras.metrics.Precision(name="precision"), 
             tf.keras.metrics.Recall(name="recall")],
)

history_fine = model.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen,
    class_weight=class_weights, 
    callbacks=callbacks,
)


[PHASE 2] Fine-tuning top layers...
Epoch 1/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9444 - auc: 0.9881 - loss: 0.1375 - precision: 0.9829 - recall: 0.9398
Epoch 1: val_auc did not improve from 0.99734
129/129 ━━━━━━━━━━━━━━━━━━━━ 222s 2s/step - accuracy: 0.9461 - auc: 0.9878 - loss: 0.1377 - precision: 0.9829 - recall: 0.9425 - val_accuracy: 0.9293 - val_auc: 0.9972 - val_loss: 0.1605 - val_precision: 0.9966 - val_recall: 0.9062 - learning_rate: 1.0000e-05
Epoch 2/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9414 - auc: 0.9860 - loss: 0.1469 - precision: 0.9824 - recall: 0.9359
Epoch 2: val_auc improved from 0.99734 to 0.99752, saving model to ./model\pneumonia_densenet.keras

Epoch 2: finished saving model to ./model\pneumonia_densenet.keras
129/129 ━━━━━━━━━━━━━━━━━━━━ 197s 2s/step - accuracy: 0.9456 - auc: 0.9874 - loss: 0.1401 - precision: 0.9849 - recall: 0.9398 - val_accuracy: 0.9316 - val_auc: 0.9975 - val_loss: 0.1570 - val_precision: 0.9983 - val_

### Evaluation

In [18]:
print("\n[EVAL] Testing on held-out test set...")
results = model.evaluate(test_gen)
print(f"Test Loss:      {results[0]:.4f}")
print(f"Test Accuracy:  {results[1]*100:.2f}%")
print(f"Test AUC:       {results[2]:.4f}")
print(f"Test Precision: {results[3]:.4f}")
print(f"Test Recall:    {results[4]:.4f}")

model.save(MODEL_PATH)
print(f"\n[SAVED] Model saved to: {MODEL_PATH}")


[EVAL] Testing on held-out test set...
28/28 ━━━━━━━━━━━━━━━━━━━━ 27s 963ms/step - accuracy: 0.9364 - auc: 0.9960 - loss: 0.1667 - precision: 0.9983 - recall: 0.9143
Test Loss:      0.1667
Test Accuracy:  93.64%
Test AUC:       0.9960
Test Precision: 0.9983
Test Recall:    0.9143

[SAVED] Model saved to: ./model\pneumonia_densenet.keras
